In [5]:

import pandas as pd
from pathlib import Path

# Step 1: Locate the dataset
file_name = "csc_wait_times_2025_06_11_1155.csv"

possible_paths = [
    Path("../data/raw") / file_name
]

file_path = next(
    (p for p in possible_paths if p.exists()),
    None
)

if file_path is None:
    raise FileNotFoundError("CSV file not found in data/raw")

# Step 2: Read the CSV
df = pd.read_csv(file_path, dtype="string")

# Step 3: Understand the dataset
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 10 records:")
display(df.head(10))


Total rows: 146219
Total columns: 4

Column names:
['Date', 'CSC_NAME', 'NUMBER_OF_CUSTOMERS', 'AVG_WAIT_TIME']

First 10 records:


,Date,CSC_NAME,NUMBER_OF_CUSTOMERS,AVG_WAIT_TIME
0,2018-07-15,Ipswich CSC,7,12:10:40
1,2018-07-16,Ipswich CSC,363,00:12:02
2,2018-07-17,Ipswich CSC,343,00:07:53
3,2018-07-17,Toowoomba City CSC,5,03:06:50
4,2018-07-18,Ipswich CSC,282,00:10:12
5,2018-07-18,Toowoomba Harristown CSC,9,00:27:55
6,2018-07-18,Toowoomba City CSC,237,00:10:14
7,2018-07-19,Ipswich CSC,330,00:09:24
8,2018-07-19,Toowoomba Harristown CSC,217,00:02:58
9,2018-07-19,Toowoomba City CSC,204,00:06:02


In [6]:

# Create a copy so the original data stays unchanged
profile = df.copy()

# Remove unnecessary spaces
for col in profile.columns:
    profile[col] = profile[col].str.strip()
    profile[col] = profile[col].replace("", pd.NA)

# Convert fields for validation
dates = pd.to_datetime(
    profile["Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

customers = pd.to_numeric(
    profile["NUMBER_OF_CUSTOMERS"],
    errors="coerce"
)

wait_times = pd.to_timedelta(
    profile["AVG_WAIT_TIME"],
    errors="coerce"
)

print("===== DATASET OVERVIEW =====")
print("Total records:", len(profile))
print("Unique service centres:", profile["CSC_NAME"].nunique())
print("Earliest date:", dates.min())
print("Latest date:", dates.max())

print("\n===== MISSING VALUES =====")
print(profile.isna().sum())

print("\n===== DUPLICATES =====")
print("Fully duplicated rows:", profile.duplicated().sum())

print("\n===== DATA VALIDATION =====")
print("Invalid/missing dates:", dates.isna().sum())
print("Invalid/missing customer counts:", customers.isna().sum())
print("Invalid/missing waiting times:", wait_times.isna().sum())
print("Negative customer counts:", (customers < 0).sum())
print("Waiting times over 2 hours:", (wait_times > pd.Timedelta(hours=2)).sum())


===== DATASET OVERVIEW =====
Total records: 146219
Unique service centres: 99
Earliest date: 2018-07-15 00:00:00
Latest date: 2025-06-10 00:00:00

===== MISSING VALUES =====
Date                   0
CSC_NAME               0
NUMBER_OF_CUSTOMERS    0
AVG_WAIT_TIME          0
dtype: int64

===== DUPLICATES =====
Fully duplicated rows: 0

===== DATA VALIDATION =====
Invalid/missing dates: 0
Invalid/missing customer counts: 0
Invalid/missing waiting times: 0
Negative customer counts: 0
Waiting times over 2 hours: 30


In [7]:

# Identify records with waiting times over 2 hours
outliers = profile[
    wait_times > pd.Timedelta(hours=2)
].copy()

# Convert waiting time into hours for easier analysis
outliers["WAIT_HOURS"] = (
    wait_times.loc[outliers.index]
    .dt.total_seconds() / 3600
).round(2)

# Sort by highest waiting time
outliers = outliers.sort_values(
    by="WAIT_HOURS",
    ascending=False
)

print("Total unusual records:", len(outliers))

print("\nTop 15 highest waiting times:")
display(outliers.head(15))

print("\nService centres with unusual waiting times:")
print(outliers["CSC_NAME"].value_counts())


Total unusual records: 30

Top 15 highest waiting times:


,Date,CSC_NAME,NUMBER_OF_CUSTOMERS,AVG_WAIT_TIME,WAIT_HOURS
0,2018-07-15,Ipswich CSC,7,12:10:40,12.18
42661,2020-09-27,Gatton CSC,1,04:27:15,4.45
910,2018-09-22,Brisbane City CSC,1,04:26:08,4.44
17824,2019-08-11,Caboolture CSC,1,04:25:24,4.42
48348,2021-01-03,Gatton CSC,1,04:24:32,4.41
140512,2025-03-06,Carseldine CSC,1,04:22:12,4.37
106952,2023-09-01,Currumbin Waters CSC,9,04:21:26,4.36
102279,2023-06-21,Redcliffe CSC,64,04:20:58,4.35
3099,2018-11-25,Macgregor CSC,2,04:19:44,4.33
88961,2022-11-07,Bundamba CSC,5,04:18:27,4.31



Service centres with unusual waiting times:
CSC_NAME
Carseldine CSC                   5
Macgregor CSC                    3
Bundamba CSC                     3
Gatton CSC                       2
Cairns CSC                       2
Ipswich CSC                      1
Brisbane City CSC                1
Caboolture CSC                   1
Currumbin Waters CSC             1
Redcliffe CSC                    1
Maroochydore CSC                 1
Gladstone CSC                    1
Redbank Select Service Centre    1
Warwick CSC                      1
Wynnum CSC                       1
Bentley Park CSC                 1
Cleveland CSC                    1
Beenleigh CSC 2                  1
Toowoomba City CSC               1
Toowong CSC                      1
Name: count, dtype: int64[pyarrow]


In [8]:

# Check duplicate combinations of Date and CSC_NAME
duplicate_keys = profile.duplicated(
    subset=["Date", "CSC_NAME"],
    keep=False
)

print("Records with repeated centre-date combinations:")
print(duplicate_keys.sum())

print("\nSample repeated records:")
display(profile[duplicate_keys].head(10))

# Investigate customer volumes in the outlier records
outlier_customers = customers.loc[outliers.index]

print("\n===== OUTLIER CUSTOMER ANALYSIS =====")
print(
    "Outliers with 5 or fewer customers:",
    (outlier_customers <= 5).sum()
)

print(
    "Outliers with more than 5 customers:",
    (outlier_customers > 5).sum()
)

print(
    "Highest customer count among outliers:",
    outlier_customers.max()
)


Records with repeated centre-date combinations:
0

Sample repeated records:


,Date,CSC_NAME,NUMBER_OF_CUSTOMERS,AVG_WAIT_TIME



===== OUTLIER CUSTOMER ANALYSIS =====
Outliers with 5 or fewer customers: 24
Outliers with more than 5 customers: 6
Highest customer count among outliers: 175


In [9]:

# Create a copy of the outlier records
high_volume_outliers = outliers.copy()

# Add the numeric customer count
high_volume_outliers["CUSTOMERS"] = customers.loc[
    high_volume_outliers.index
]

# Keep outliers with more than 5 customers
high_volume_outliers = high_volume_outliers[
    high_volume_outliers["CUSTOMERS"] > 5
]

# Sort by customer count
high_volume_outliers = high_volume_outliers.sort_values(
    by="CUSTOMERS",
    ascending=False
)

print("Outliers with more than 5 customers:")
display(
    high_volume_outliers[
        ["Date", "CSC_NAME", "CUSTOMERS",
         "AVG_WAIT_TIME", "WAIT_HOURS"]
    ]
)


Outliers with more than 5 customers:


,Date,CSC_NAME,CUSTOMERS,AVG_WAIT_TIME,WAIT_HOURS
1032,2018-09-27,Toowong CSC,175,03:01:16,3.02
102279,2023-06-21,Redcliffe CSC,64,04:20:58,4.35
10,2018-07-19,Warwick CSC,13,04:09:44,4.16
119059,2024-03-21,Cairns CSC,10,02:14:46,2.25
106952,2023-09-01,Currumbin Waters CSC,9,04:21:26,4.36
0,2018-07-15,Ipswich CSC,7,12:10:40,12.18


In [11]:

# Function to investigate one unusual waiting-time record

def investigate_outlier(centre, target_date, days=7):

    target_date = pd.Timestamp(target_date)

    # Define the investigation period
    start_date = target_date - pd.Timedelta(days=days)
    end_date = target_date + pd.Timedelta(days=days)

    # Select records for this centre and date range
    nearby = analysis[
        (analysis["CSC_NAME"] == centre) &
        (analysis["Date"] >= start_date) &
        (analysis["Date"] <= end_date)
    ].copy()

    # Separate the unusual record from surrounding dates
    target = nearby[nearby["Date"] == target_date]
    surrounding = nearby[nearby["Date"] != target_date]

    if target.empty:
        return None

    # Compare waiting times around the incident
    median_all = surrounding["WAIT_MINUTES"].median()

    positive_waits = surrounding[
        surrounding["WAIT_MINUTES"] > 0
    ]["WAIT_MINUTES"]

    median_positive = positive_waits.median()

    zero_wait_days = (
        surrounding["WAIT_MINUTES"] == 0
    ).sum()

    # Return a summary of the investigation
    return {
        "Centre": centre,
        "Date": target_date.date(),
        "Customers": target["CUSTOMERS"].iloc[0],
        "Incident Wait (min)": round(
            target["WAIT_MINUTES"].iloc[0], 2
        ),
        "Nearby Records": len(surrounding),
        "Nearby Zero-Wait Records": zero_wait_days,
        "Nearby Median (min)": round(median_all, 2),
        "Positive-Wait Median (min)": round(
            median_positive, 2
        )
    }


# Run the function for all six high-volume outliers
results = []

for _, row in high_volume_outliers.iterrows():

    result = investigate_outlier(
        row["CSC_NAME"],
        row["Date"]
    )

    if result is not None:
        results.append(result)


# Create a summary table
investigation_summary = pd.DataFrame(results)

investigation_summary = investigation_summary.sort_values(
    "Incident Wait (min)",
    ascending=False
)

print("AUTOMATED OUTLIER INVESTIGATION RESULTS")
display(investigation_summary)


AUTOMATED OUTLIER INVESTIGATION RESULTS


,Centre,Date,Customers,Incident Wait (min),Nearby Records,Nearby Zero-Wait Records,Nearby Median (min),Positive-Wait Median (min)
5,Ipswich CSC,2018-07-15,7,730.67,5,0,9.40,9.40
4,Currumbin Waters CSC,2023-09-01,9,261.43,10,0,9.95,9.95
1,Redcliffe CSC,2023-06-21,64,260.97,10,1,22.33,23.12
2,Warwick CSC,2018-07-19,13,249.73,5,0,4.13,4.13
0,Toowong CSC,2018-09-27,175,181.27,9,5,0.00,5.24
3,Cairns CSC,2024-03-21,10,134.77,8,0,17.02,17.02


In [12]:

# Step 8: Investigate zero waiting times

# Find records where waiting time is exactly zero
zero_wait_records = analysis[
    analysis["WAIT_MINUTES"] == 0
].copy()

print("===== ZERO WAIT TIME ANALYSIS =====")

print("Total records:", len(analysis))

print(
    "Records with zero waiting time:",
    len(zero_wait_records)
)

print(
    "Percentage of records with zero waiting time:",
    round(len(zero_wait_records) / len(analysis) * 100, 2),
    "%"
)

print(
    "Zero-wait records with customers > 0:",
    (zero_wait_records["CUSTOMERS"] > 0).sum()
)

print(
    "Zero-wait records with 100+ customers:",
    (zero_wait_records["CUSTOMERS"] >= 100).sum()
)

# Examine high-volume zero-wait records
print("\nTop 10 zero-wait records by customer volume:")

display(
    zero_wait_records.sort_values(
        "CUSTOMERS",
        ascending=False
    )[
        ["Date", "CSC_NAME", "CUSTOMERS", "WAIT_MINUTES"]
    ].head(10)
)

# Check how zero waiting times vary by year
zero_wait_records["Year"] = (
    zero_wait_records["Date"].dt.year
)

print("\nZero-wait records by year:")

display(
    zero_wait_records.groupby("Year")
    .size()
    .reset_index(name="Zero_Wait_Records")
)


===== ZERO WAIT TIME ANALYSIS =====
Total records: 146219
Records with zero waiting time: 77277
Percentage of records with zero waiting time: 52.85 %
Zero-wait records with customers > 0: 77272
Zero-wait records with 100+ customers: 6785

Top 10 zero-wait records by customer volume:


,Date,CSC_NAME,CUSTOMERS,WAIT_MINUTES
24665,2019-11-29,Maroochydore CSC,329,0.0
29831,2020-02-28,Carseldine CSC,272,0.0
108630,2023-09-28,Helensvale CSC,255,0.0
30011,2020-03-03,Carseldine CSC,249,0.0
792,2018-09-18,Brisbane City CSC,244,0.0
29433,2020-02-24,Carseldine CSC,241,0.0
7898,2019-02-26,Bundaberg CSC,240,0.0
29602,2020-02-26,Carseldine CSC,232,0.0
807,2018-09-19,Brisbane City CSC,227,0.0
12797,2019-05-21,Maroochydore CSC,223,0.0



Zero-wait records by year:


,Year,Zero_Wait_Records
0,2018,1872
1,2019,11737
2,2020,11895
3,2021,11814
4,2022,11516
5,2023,11620
6,2024,11780
7,2025,5043


In [13]:

# Step 9: Analyse zero-wait patterns

zero_profile = analysis.copy()

# Create investigation columns
zero_profile["Year"] = zero_profile["Date"].dt.year

zero_profile["IS_ZERO_WAIT"] = (
    zero_profile["WAIT_MINUTES"] == 0
)

# Analyse by year
year_summary = zero_profile.groupby("Year").agg(
    Total_Records=("IS_ZERO_WAIT", "size"),
    Zero_Wait_Records=("IS_ZERO_WAIT", "sum")
).reset_index()

year_summary["Zero_Wait_Percentage"] = (
    year_summary["Zero_Wait_Records"]
    / year_summary["Total_Records"] * 100
).round(2)

print("ZERO WAITING TIMES BY YEAR")
display(year_summary)

# Analyse by service centre
centre_summary = zero_profile.groupby("CSC_NAME").agg(
    Total_Records=("IS_ZERO_WAIT", "size"),
    Zero_Wait_Records=("IS_ZERO_WAIT", "sum")
).reset_index()

centre_summary["Zero_Wait_Percentage"] = (
    centre_summary["Zero_Wait_Records"]
    / centre_summary["Total_Records"] * 100
).round(2)

# Compare centres with at least 100 records
centre_summary = centre_summary[
    centre_summary["Total_Records"] >= 100
]

print("\nTOP 10 CENTRES BY ZERO-WAIT PERCENTAGE")

display(
    centre_summary.sort_values(
        "Zero_Wait_Percentage",
        ascending=False
    ).head(10)
)


ZERO WAITING TIMES BY YEAR


,Year,Total_Records,Zero_Wait_Records,Zero_Wait_Percentage
0,2018,4572,1872,40.94
1,2019,21730,11737,54.01
2,2020,22045,11895,53.96
3,2021,22059,11814,53.56
4,2022,21653,11516,53.18
5,2023,22052,11620,52.69
6,2024,22497,11780,52.36
7,2025,9611,5043,52.47



TOP 10 CENTRES BY ZERO-WAIT PERCENTAGE


,CSC_NAME,Total_Records,Zero_Wait_Records,Zero_Wait_Percentage
0,Atherton CSC,1692,1692,100.0
29,Clermont QGAP,1562,1562,100.0
32,Cunnamulla QGAP,1405,1405,100.0
34,Dalby CSC,1703,1703,100.0
35,Emerald CSC,1713,1713,100.0
39,Goondiwindi CSC,1690,1690,100.0
46,Innisfail CSC,1694,1694,100.0
49,Kingaroy CSC,1709,1709,100.0
52,Logan City CSC 2,433,433,100.0
53,Longreach CSC,1706,1706,100.0


In [14]:

# Step 10: Classify service centres

centre_quality = analysis.copy()

# Flag records reporting zero waiting time
centre_quality["IS_ZERO_WAIT"] = (
    centre_quality["WAIT_MINUTES"] == 0
)

# Summarise each service centre
centre_quality = centre_quality.groupby(
    "CSC_NAME"
).agg(
    Total_Records=("Date", "size"),
    Total_Customers=("CUSTOMERS", "sum"),
    Zero_Wait_Records=("IS_ZERO_WAIT", "sum")
).reset_index()

# Calculate zero-wait percentage
centre_quality["Zero_Wait_Percentage"] = (
    centre_quality["Zero_Wait_Records"]
    / centre_quality["Total_Records"] * 100
).round(2)

# Classify each service centre
centre_quality["Category"] = "Mixed"

centre_quality.loc[
    centre_quality["Zero_Wait_Records"]
    == centre_quality["Total_Records"],
    "Category"
] = "All zero"

centre_quality.loc[
    centre_quality["Zero_Wait_Records"] == 0,
    "Category"
] = "No zeros"

# Summary across categories
category_summary = centre_quality.groupby(
    "Category"
).agg(
    Number_of_Centres=("CSC_NAME", "count"),
    Total_Records=("Total_Records", "sum"),
    Total_Customers=("Total_Customers", "sum")
).reset_index()

print("SERVICE CENTRE REPORTING CATEGORIES")
display(category_summary)

print("\nSERVICE CENTRE DETAILS")
display(
    centre_quality.sort_values(
        "Zero_Wait_Percentage",
        ascending=False
    ).head(15)
)


SERVICE CENTRE REPORTING CATEGORIES


,Category,Number_of_Centres,Total_Records,Total_Customers
0,All zero,39,55236,2471424
1,Mixed,36,56115,10800456
2,No zeros,24,34868,9303940



SERVICE CENTRE DETAILS


,CSC_NAME,Total_Records,Total_Customers,Zero_Wait_Records,Zero_Wait_Percentage,Category
0,Atherton CSC,1692,168141,1692,100.0,All zero
31,Cloncurry CSC,1678,35005,1678,100.0,All zero
34,Dalby CSC,1703,165369,1703,100.0,All zero
35,Emerald CSC,1713,175002,1713,100.0,All zero
39,Goondiwindi CSC,1690,101625,1690,100.0,All zero
46,Innisfail CSC,1694,150394,1694,100.0,All zero
48,Julia Creek QGAP,867,3985,867,100.0,All zero
52,Logan City CSC 2,433,20987,433,100.0,All zero
53,Longreach CSC,1706,34814,1706,100.0,All zero
56,Mareeba CSC,1693,165543,1693,100.0,All zero


In [15]:

# STEP 11: Create an analysis-ready dataset

# Start with the converted dataset
cleaned_data = analysis.copy()

# Add a zero waiting-time flag
cleaned_data["ZERO_WAIT_FLAG"] = (
    cleaned_data["WAIT_MINUTES"] == 0
)

# Add a long waiting-time investigation flag
cleaned_data["LONG_WAIT_FLAG"] = (
    cleaned_data["WAIT_MINUTES"] > 120
)

# Add service-centre reporting categories
centre_categories = centre_quality.set_index(
    "CSC_NAME"
)["Category"]

cleaned_data["CENTRE_CATEGORY"] = (
    cleaned_data["CSC_NAME"].map(centre_categories)
)

# Select the columns for our reporting dataset
cleaned_data = cleaned_data[
    [
        "Date",
        "CSC_NAME",
        "CUSTOMERS",
        "AVG_WAIT_TIME",
        "WAIT_MINUTES",
        "ZERO_WAIT_FLAG",
        "LONG_WAIT_FLAG",
        "CENTRE_CATEGORY"
    ]
]

# Validate before saving
assert len(cleaned_data) == len(df)
assert cleaned_data["CENTRE_CATEGORY"].notna().all()

# Save into the processed folder
output_folder = file_path.parent.parent / "processed"
output_folder.mkdir(parents=True, exist_ok=True)

output_file = (
    output_folder / "csc_wait_times_analysis_ready.csv"
)

cleaned_data.to_csv(output_file, index=False)

print("Analysis-ready dataset saved!")
print("Location:", output_file)
print("Total records:", len(cleaned_data))

display(cleaned_data.head(10))


Analysis-ready dataset saved!
Location: ../data/processed/csc_wait_times_analysis_ready.csv
Total records: 146219


,Date,CSC_NAME,CUSTOMERS,AVG_WAIT_TIME,WAIT_MINUTES,ZERO_WAIT_FLAG,LONG_WAIT_FLAG,CENTRE_CATEGORY
0,2018-07-15,Ipswich CSC,7,12:10:40,730.666667,False,True,No zeros
1,2018-07-16,Ipswich CSC,363,00:12:02,12.033333,False,False,No zeros
2,2018-07-17,Ipswich CSC,343,00:07:53,7.883333,False,False,No zeros
3,2018-07-17,Toowoomba City CSC,5,03:06:50,186.833333,False,True,No zeros
4,2018-07-18,Ipswich CSC,282,00:10:12,10.200000,False,False,No zeros
5,2018-07-18,Toowoomba Harristown CSC,9,00:27:55,27.916667,False,False,No zeros
6,2018-07-18,Toowoomba City CSC,237,00:10:14,10.233333,False,False,No zeros
7,2018-07-19,Ipswich CSC,330,00:09:24,9.400000,False,False,No zeros
8,2018-07-19,Toowoomba Harristown CSC,217,00:02:58,2.966667,False,False,No zeros
9,2018-07-19,Toowoomba City CSC,204,00:06:02,6.033333,False,False,No zeros


In [16]:

# STEP 12: Validate the exported dataset

# Read the processed CSV
validated_data = pd.read_csv(output_file)

# Check rows and columns
print("Original rows:", len(df))
print("Processed rows:", len(validated_data))

print("\nProcessed columns:")
print(validated_data.columns.tolist())

# Check missing values
print("\nMissing values:")
print(validated_data.isna().sum())

# Check data quality flags
print("\nZero-wait records:")
print(validated_data["ZERO_WAIT_FLAG"].sum())

print("\nLong-wait records:")
print(validated_data["LONG_WAIT_FLAG"].sum())

# Validate expected results
assert len(validated_data) == 146219
assert len(validated_data.columns) == 8
assert validated_data["Date"].notna().all()
assert validated_data["ZERO_WAIT_FLAG"].sum() == 77277
assert validated_data["LONG_WAIT_FLAG"].sum() == 30

print("\nDATA VALIDATION PASSED!")


Original rows: 146219
Processed rows: 146219

Processed columns:
['Date', 'CSC_NAME', 'CUSTOMERS', 'AVG_WAIT_TIME', 'WAIT_MINUTES', 'ZERO_WAIT_FLAG', 'LONG_WAIT_FLAG', 'CENTRE_CATEGORY']

Missing values:
Date               0
CSC_NAME           0
CUSTOMERS          0
AVG_WAIT_TIME      0
WAIT_MINUTES       0
ZERO_WAIT_FLAG     0
LONG_WAIT_FLAG     0
CENTRE_CATEGORY    0
dtype: int64

Zero-wait records:
77277

Long-wait records:
30

DATA VALIDATION PASSED!
